# Sentiment Analysis: Vanilla LSTM vs. Attention-Augmented LSTM

Showing how adding an **Attention Mechanism** to an LSTM improves interpretability and performance by allowing the model to focus on specific keywords in a sentence.

## 1. Data Preparation
We use a toy dataset of positive and negative sentences.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

In [4]:
# ToyTest Dataset
data = [
    ("I love this movie", 1),
    ("This film was amazing", 1),
    ("Great acting and plot", 1),
    ("I hated this movie", 0),
    ("Worst film ever", 0),
    ("Terrible and boring", 0),
    ("The food was good but service was slow", 0), # Mixed signal
    ("Amazing experience all around", 1)
]


In [5]:
# Simple Tokenization
vocab = {"<PAD>": 0}
for sent, _ in data:
    for word in sent.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

def prepare_sequence(seq, vocab):
    idxs = [vocab[w] for w in seq.lower().split()]
    return torch.tensor(idxs, dtype=torch.long)



In [6]:
print(f"Vocab Size: {len(vocab)}")
print(vocab)

Vocab Size: 26
{'<PAD>': 0, 'i': 1, 'love': 2, 'this': 3, 'movie': 4, 'film': 5, 'was': 6, 'amazing': 7, 'great': 8, 'acting': 9, 'and': 10, 'plot': 11, 'hated': 12, 'worst': 13, 'ever': 14, 'terrible': 15, 'boring': 16, 'the': 17, 'food': 18, 'good': 19, 'but': 20, 'service': 21, 'slow': 22, 'experience': 23, 'all': 24, 'around': 25}


## 2. Models

### Vanilla LSTM
Takes the **final hidden state** of the sequence and passes it to a linear layer.

In [7]:
class VanillaLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(VanillaLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(0) # Batch size 1
        _, (hidden, _) = self.lstm(x)
        # Use only the last hidden state: hidden index 0 since num_layers=1
        out = self.fc(hidden[-1])
        return out

### Attention-Augmented LSTM
Calculates a weighted average of **all hidden states** based on their importance.

In [8]:
class AttentionLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(AttentionLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        # Attention components
        self.attn_weights_layer = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(0)
        lstm_out, _ = self.lstm(x) # (1, seq_len, hidden_dim)
        
        # 1. Calculate Attention Scores
        attn_scores = self.attn_weights_layer(lstm_out) # (1, seq_len, 1)
        
        # 2. Softmax to get weights
        attn_weights = F.softmax(attn_scores, dim=1)
        
        # 3. Context Vector (Weighted Sum of hidden states)
        context_vector = torch.sum(attn_weights * lstm_out, dim=1)
        
        out = self.fc(context_vector)
        return out, attn_weights

## 3. Training

Train both models on the toy dataset.

In [9]:
def train_model(model, data, epochs=50):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        total_loss = 0
        for sent, label in data:
            model.zero_grad()
            inputs = prepare_sequence(sent, vocab)
            target = torch.tensor([label], dtype=torch.long)
            
            # Handle models with/without attention weights return
            output = model(inputs)
            if isinstance(output, tuple): output = output[0]
            
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

vanilla_model = VanillaLSTM(len(vocab), 16, 16)
attn_model = AttentionLSTM(len(vocab), 16, 16)

print("Training Vanilla LSTM...")
train_model(vanilla_model, data)
print("\nTraining Attention LSTM...")
train_model(attn_model, data)

Training Vanilla LSTM...
Epoch 10, Loss: 0.0385
Epoch 20, Loss: 0.0083
Epoch 30, Loss: 0.0045
Epoch 40, Loss: 0.0029
Epoch 50, Loss: 0.0020

Training Attention LSTM...
Epoch 10, Loss: 0.0452
Epoch 20, Loss: 0.0104
Epoch 30, Loss: 0.0054
Epoch 40, Loss: 0.0033
Epoch 50, Loss: 0.0023


## 4. Why Attention? (The Big Difference)

Students often ask: **"If the LSTM already has memory, why do we need Attention?"**

### The Vanilla LSTM Bottleneck
In a standard LSTM, the model processes words one by one. By the time it reaches the end of a sentence, it tries to squeeze the *entire meaning* into the very last hidden state. 
- **The Problem**: If the sentence is long, the model might "forget" important words from the beginning. This is called the information bottleneck.
- **The Blindness**: We cannot see which words the model actually prioritized. It's a "black box".

### The Attention Spotlight
Attention removes this bottleneck by keeping **all hidden states** available. It then calculates a "weight" (importance) for each word.
- **Dynamic Access**: The model can "peek" back at highly emotional words like *amazing* or *terrible* regardless of where they appear.
- **Interpretability**: The bar charts below are the "proof" of the model's reasoning. We can see exactly which word triggered the positive or negative sentiment (e.g., notice how 'love' gets a high bar regardless of sentence position).

## 5. Visualizing Attention

The bar charts below show the "attention scores". Words with higher weights contributed more to the final classification.

In [10]:
def plot_attention(sentence, model):
    model.eval()
    inputs = prepare_sequence(sentence, vocab)
    _, weights = model(inputs)
    
    weights = weights.squeeze().detach().numpy()
    words = sentence.split()
    
    plt.figure(figsize=(10, 2))
    plt.bar(words, weights)
    plt.title(f"Attention weights for: '{sentence}'")
    plt.ylabel("Weight")
    plt.show()

test_sentence = "I love this amazing movie"
plot_attention(test_sentence, attn_model)

test_sentence_2 = "This film was boring and hated"
plot_attention(test_sentence_2, attn_model)